# Framing Annotation — Step 2: Improve Coverage of Draft Annotation Instructions

- **Step 2.1** — Test draft annotation instructions on a small random sample and inspect results  
- **Step 2.2** — Run multiple annotation runs (briseus equivalent) to identify hard cases where the model disagrees with itself

The goal at this stage is **not** to achieve high reliability yet — it is to make sure your annotation instructions cover all cases you encounter in the data. Hard cases and mislabeled cases will tell you where to add decision rules and clarify your codebook.

## Setup

Load libraries and configure the API connection. The API key is loaded from a `.env` file — make sure you have a `.env` file in your project root with:

```
CAC_API_KEY=your_key_here
```

In [22]:
import pandas as pd
import requests
import os
import time
from dotenv import load_dotenv
from pathlib import Path 
from collections import Counter

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

print(f"API key loaded: {'YES' if api_key else 'NO — check your .env file'}")
print(f"Host: {HOST}")
print(f"Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1
Model: ministral-3-14b


In [2]:
!echo $CAC_API_KEY

sk-T4QZ6ao-ia3njvfff13KEQ


## Step 1 Draft: Annotation Instructions

These are the **Step 1 annotation instructions** written based on the theoretical framework from Freudenthaler et al. (under review). The **Immigration and Crime Frame** (Security Threat Frame in the paper) belongs to dimension 5 of the framework — the ideological layer that links structural inequalities to attitudes and behaviors.

According to the paper, security threat frames portray racialized groups as implicitly or explicitly causing crime, justifying harsher policing and restrictive immigration policies.
elated to group 1 !! 
> **Note:** These instructions are a first draft and will be refined iteratively through Steps 2.1 and 2.2. Modify the `instructions` and `reminder` variables below after each round of inspection.

## NEW INSTRUCTION! 

In [42]:
instructions = """Sie sind ein Experte für Inhaltsanalyse in einer Studie zur medialen Darstellung ethnischer und sozialer Gruppen in deutschen Nachrichten. Ihre Aufgabe ist es, zu klassifizieren, ob ein gegebener Nachrichtenabsatz einen Einwanderungs- und Kriminalitätsrahmen enthält und, falls ja, ob dieser negativ (Bedrohung) oder positiv (Sicherheitsgewinn/Prävention) dargestellt wird.

Sie erhalten Mehrfachabsätze aus deutschen Zeitungsartikeln. Gruppen können als [Gruppe 1] oder [andere Gruppe] bezeichnet werden. Konzentrieren Sie sich ausschließlich auf das, was explizit im Text steht. Schließen Sie keine Gruppenassoziationen, die nicht direkt ausgedrückt werden.

WICHTIG:
Ein Kriminalitäts- oder Sicherheitsrahmen liegt nur dann vor, wenn die Gruppe explizit mit Kriminalität, Sicherheit, Gefahr oder Prävention verknüpft wird.

NO_CRIME_FRAME:
Ein Absatz erhält dieses Label, wenn kein expliziter Kriminalitäts- oder Sicherheitsrahmen vorliegt. Dies gilt insbesondere wenn:
- Die Gruppe gar nicht erwähnt wird
- Kriminalität erwähnt wird, aber nicht mit der Gruppe verknüpft ist
- Die Gruppe als Opfer von Straftaten dargestellt wird
- Kriminalität auf soziale Bedingungen (Armut, Ausgrenzung etc.) zurückgeführt wird
- Der Text Kriminalitätszuschreibungen relativiert oder widerspricht („kein Generalverdacht“, „nicht alle“)
- Krieg, geopolitische Konflikte oder militärische Ereignisse beschrieben werden, ohne explizite Verknüpfung der Gruppe mit Terrorismus, Anschlägen, Gewalt gegen Zivilisten, Straftaten oder Sicherheitsbedrohungen
- Humanitäre Krisen oder Naturkatastrophen beschrieben werden
- Ein Täter erwähnt wird, aber NICHT Mitglied der Gruppe ist

CRIME_FRAME_NEG:
Ein Absatz erhält dieses Label, wenn die Gruppe als Quelle von Kriminalität oder Sicherheitsbedrohung dargestellt wird. Dies umfasst:
- Gruppe wird als Täter, Verdächtiger oder kriminell dargestellt
- Gruppe wird mit konkreten Straftaten verbunden (z.B. Diebstahl, Gewalt, Mord)
- Gruppe wird als Ursache steigender Kriminalität dargestellt
- Gruppe wird als Gefahr für öffentliche oder nationale Sicherheit dargestellt
- Migration oder Gruppenzugehörigkeit wird mit Kriminalität verknüpft
- Terrorismus oder Extremismus wird mit der Gruppe verbunden
- Physische Gewalt oder Bedrohung geht explizit von Gruppenmitgliedern aus
- Gruppe begrüßt, legitimiert, feiert, rechtfertigt oder unterstützt einen Anschlag, Terrorakt, Gewaltakt oder eine Straftat
- Gruppe stellt einen Anschlag, Terrorakt, Gewaltakt oder eine Straftat als Widerstand, heroische Operation oder legitime Handlung dar
- Gruppe erscheint als Unterstützer, Kommentator, ideologischer Bezugspunkt oder politischer Akteur, der explizit mit Zustimmung, Rechtfertigung, Feier, Unterstützung oder ideologischer Nähe zu Terrorismus, politischer Gewalt oder Straftaten verbunden wird

WICHTIG:
Die Gruppe muss klar mit der Straftat, Bedrohung, Gewaltlegitimation oder Sicherheitsgefahr verknüpft sein. Reine Erwähnung von Kriminalität reicht nicht. Eine direkte Verantwortung oder Täterschaft der Gruppe ist nicht zwingend erforderlich, wenn die Gruppe den Gewaltakt ausdrücklich legitimiert, begrüßt, feiert oder unterstützt.

CRIME_FRAME_POS:
Ein Absatz erhält dieses Label, wenn ein Sicherheits- oder Kriminalitätsrahmen vorhanden ist, aber positiv dargestellt wird. Dies umfasst:
- Maßnahmen zur Kriminalitätsbekämpfung im Zusammenhang mit der Gruppe
- Prävention (z.B. soziale Arbeit, Integration, Deradikalisierung)
- Differenzierung („kein Generalverdacht“, „nicht alle“)
- Kriminalität wird durch soziale Faktoren erklärt statt durch die Gruppe selbst
- Gruppenmitglieder wirken aktiv an Sicherheit oder Prävention mit (z.B. Polizei, Sozialarbeit)
- Maßnahmen werden als Verbesserung von Sicherheit oder Ordnung dargestellt

Entscheidungsregeln:
- Prüfen Sie zuerst, ob überhaupt ein Kriminalitäts- oder Sicherheitsrahmen vorliegt
- Wenn nein → NO_CRIME_FRAME
- Wenn ja → prüfen, ob die Darstellung negativ (Bedrohung) oder positiv (Schutz/Prävention) ist
- Wenn Gruppe als Täter, Gefahr, Unterstützer von Gewalt oder Legitimierer von Terrorismus erscheint → CRIME_FRAME_NEG
- Wenn Krieg, geopolitische Konflikte oder militärische Ereignisse vorkommen, vergeben Sie nur dann CRIME_FRAME_NEG, wenn die Gruppe explizit mit Terrorismus, Anschlägen, Gewalt gegen Zivilisten, Straftaten oder Sicherheitsbedrohungen verbunden wird
- Wenn Sicherheit, Prävention oder Differenzierung im Vordergrund steht → CRIME_FRAME_POS
- Wenn unklar oder widersprüchlich → UNCLEAR

Im Zweifelsfall:
Fragen Sie sich, ob ein Leser die Gruppe als kriminell, sicherheitsgefährdend, gewaltunterstützend oder als Sicherheitsfaktor wahrnehmen würde. Wenn nein → NO_CRIME_FRAME."""

reminder = "Vergeben Sie genau ein Label: NO_CRIME_FRAME, CRIME_FRAME_NEG oder CRIME_FRAME_POS. Achten Sie besonders darauf, dass die Gruppe explizit mit Kriminalität, Sicherheit, Terrorismus, Gewaltlegitimation oder Prävention verknüpft ist. Wenn Kriminalität erwähnt wird, aber nicht mit der Gruppe verbunden ist oder die Gruppe Opfer ist, vergeben Sie NO_CRIME_FRAME."

print("Instructions and reminder loaded.")

Instructions and reminder loaded.


Test another Instruction 

## Core Functions

Three functions power the annotation pipeline:

- `annotate()` — sends a single paragraph to the API and returns the raw model output
- `parse_label()` — extracts the label from the raw output, handling misspellings and drift
- `parse_explanation()` — extracts the explanation sentence from the raw output

**On label drift:** Small LLMs sometimes output unexpected labels (e.g. `CRIME` instead of `CRIME_FRAME`, or a misspelling). The `parse_label()` function handles this by doing a fuzzy match rather than an exact match. If more than 5% of your outputs are `UNCLEAR`, revisit the reminder and output format instructions.

In [43]:
def annotate(text, instructions, reminder, temperature=0.0):
    """Send a paragraph to the API for annotation."""
    prompt = f"{instructions}\n\nNow annotate this paragraph:\n\n{text}\n\n{reminder}\n\nRespond in this format exactly:\nLabel: <NO_CRIME_FRAME or CRIME_FRAME_NEG or CRIME_FRAME_POS>\nExplanation: <one sentence explaining your choice>"

    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": "Sie sind ein Experte für Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen genau und antworten Sie immer im angegebenen Format."},
            {"role": "user", "content": prompt}
        ]
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(3):
        try:
            response = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if response.status_code == 200:
                return response.json()['choices'][0]['message']['content'].strip()
            else:
                print(f"  [attempt {attempt+1}] Status {response.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)
    return "ERROR"


def parse_label(raw_output):
    """Extract and normalize label from raw model output.
    Handles misspellings and label drift from small LLMs.
    """
    raw_upper = raw_output.upper()

    # Check NO_CRIME_FRAME first because it contains CRIME_FRAME
    if "NO_CRIME_FRAME" in raw_upper or "NO CRIME FRAME" in raw_upper:
        return "NO_CRIME_FRAME"
    elif "CRIME_FRAME_NEG" in raw_upper or "CRIME FRAME NEG" in raw_upper:
        return "CRIME_FRAME_NEG"
    elif "CRIME_FRAME_POS" in raw_upper or "CRIME FRAME POS" in raw_upper:
        return "CRIME_FRAME_POS"
    else:
        return "UNCLEAR"


def parse_explanation(raw_output):
    """Extract explanation sentence from raw model output."""
    for line in raw_output.split("\n"):
        if line.lower().startswith("explanation:"):
            return line[len("explanation:"):].strip()
    return raw_output


print("Functions loaded.")

Functions loaded.


## Step 2.1 — Test Draft Annotation Instructions

We run the annotation on a **random sample of 50 rows** from the translated dataset. This is equivalent to the `bacchuss()` call in the R vignette.

After running, visually inspect the results and note:
- Which cases were correctly labelled?
- Are there clear cases not described in your coding instructions?
- What are general problems with the instructions?

Then go back and update the `instructions` and `reminder` in the cell above.

In [44]:
# ── Config ───────────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

SAMPLE_SIZE_2_1 = 20000
RANDOM_SEED = 42

# Saving of full run result
OUTPUT_2_1 = RESULTS_DIR / f"annotation_step2_1_seed{RANDOM_SEED}_n{SAMPLE_SIZE_2_1}.csv"

# Separate continuous files for relevant cases makes it easier to define test set 
CRIME_FRAME_NEG = RESULTS_DIR / "crime_frame_neg.csv"
CRIME_FRAME_POS = RESULTS_DIR / "crime_frame_pos.csv"
UNCLEAR_FRAME = RESULTS_DIR / "unclear_frame.csv"

UNCLEAR_LABELS = ["UNCLEAR"]

# Manual testset later (n=500)
MANUAL_TESTSET = RESULTS_DIR / "manual_testset.csv"
# ─────────────────────────────────────────────────────────────────────────

df = pd.read_csv("dataset/all_multi_paragraphs_2022_2026.csv")
sample_2_1 = df.sample(n=SAMPLE_SIZE_2_1, random_state=RANDOM_SEED).copy().reset_index(drop=True)

print(f"Dataset loaded {len(df)} rows total")
print(f"Sample size {len(sample_2_1)} rows")
print("Running annotation at temperature=0.0 (deterministic)\n")

Dataset loaded 659895 rows total
Sample size 20000 rows
Running annotation at temperature=0.0 (deterministic)



In [45]:
# Sample from the full dataset
sample_2_1 = df.sample(
    n=SAMPLE_SIZE_2_1,
    random_state=RANDOM_SEED
).reset_index(drop=True)

In [46]:
results_2_1 = []

for i, row in sample_2_1.iterrows():
    raw = annotate(str(row["text_block"]), instructions, reminder, temperature=0.0)
    label = parse_label(raw)
    explanation = parse_explanation(raw)

    results_2_1.append({
        "article_id":    row["article_id"],
        "par_index":     row["par_index"],
        "group":         row["group"],
        "text_block_en": row["text_block"],
        "raw_output":    raw,
        "label":         label,
        "explanation":   explanation
    })

    if (i + 1) % 10 == 0:
        n_neg = sum(1 for r in results_2_1 if r["label"] == "CRIME_FRAME_NEG")
        n_pos = sum(1 for r in results_2_1 if r["label"] == "CRIME_FRAME_POS")
        n_unclear = sum(1 for r in results_2_1 if r["label"] == "UNCLEAR")

        print(
            f"  [{i+1}/{len(sample_2_1)}] done "
            f"NEG: {n_neg}, POS: {n_pos}, UNCLEAR: {n_unclear}"
        )

annotation_2_1 = pd.DataFrame(results_2_1)

# Save full Step 2.1 output only
annotation_2_1.to_csv(OUTPUT_2_1, index=False)

print(f"\nSaved Step 2.1 to {OUTPUT_2_1}")
print("\nDone!")
print(annotation_2_1["label"].value_counts())

crime_total = annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS"]).sum()

print(f"\nAny CRIME_FRAME rate: {crime_total / len(annotation_2_1) * 100:.1f}%")
print(f"CRIME_FRAME_NEG rate: {(annotation_2_1['label'] == 'CRIME_FRAME_NEG').mean() * 100:.1f}%")
print(f"CRIME_FRAME_POS rate: {(annotation_2_1['label'] == 'CRIME_FRAME_POS').mean() * 100:.1f}%")
print(f"NO_CRIME_FRAME rate: {(annotation_2_1['label'] == 'NO_CRIME_FRAME').mean() * 100:.1f}%")
print(f"UNCLEAR rate: {(annotation_2_1['label'] == 'UNCLEAR').mean() * 100:.1f}%")

  [10/20000] done NEG: 1, POS: 0, UNCLEAR: 0
  [20/20000] done NEG: 2, POS: 0, UNCLEAR: 0
  [30/20000] done NEG: 2, POS: 0, UNCLEAR: 0
  [40/20000] done NEG: 3, POS: 0, UNCLEAR: 0
  [50/20000] done NEG: 3, POS: 0, UNCLEAR: 0
  [60/20000] done NEG: 4, POS: 0, UNCLEAR: 0
  [70/20000] done NEG: 4, POS: 0, UNCLEAR: 0
  [80/20000] done NEG: 6, POS: 0, UNCLEAR: 0
  [90/20000] done NEG: 7, POS: 0, UNCLEAR: 0
  [100/20000] done NEG: 8, POS: 0, UNCLEAR: 0
  [110/20000] done NEG: 8, POS: 0, UNCLEAR: 0
  [120/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [130/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [140/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [150/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [160/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [170/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [180/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [190/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [200/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [210/20000] done NEG: 9, POS: 0, UNCLEAR: 0
  [220/20000] done NEG: 10, POS: 0, UNCLEAR

### Inspect Results

Check the label distribution and flag any `UNCLEAR` labels (label drift). If more than **5% of labels are UNCLEAR**, adjust the reminder and output format instructions and rerun.

Use the output below to visually inspect cases — especially CRIME_FRAME cases — and ask:
- Does the label match what you would expect?
- Are there edge cases your instructions don't cover yet?

In [47]:
# Label distribution
total = len(annotation_2_1)
print("=== Label Distribution ===")
for label, count in annotation_2_1["label"].value_counts().items():
    print(f"  {label}: {count} ({count/total*100:.1f}%)")

# Flag unclear labels
unclear = annotation_2_1[annotation_2_1["label"] == "UNCLEAR"]

if len(unclear) > 0:
    pct = len(unclear) / total * 100
    print(f"\nUNCLEAR labels: {len(unclear)} ({pct:.1f}%)")

    if pct > 5:
        print("   → More than 5% unclear. Consider adjusting the reminder and output format.")
else:
    print("\n✓ No unclear labels.")

=== Label Distribution ===
  NO_CRIME_FRAME: 19144 (95.7%)
  CRIME_FRAME_NEG: 839 (4.2%)
  CRIME_FRAME_POS: 17 (0.1%)

✓ No unclear labels.


In [48]:
# Browse all results — sort by label for easier inspection
annotation_2_1[
    ["group", "label", "explanation", "text_block_en"]
].sort_values("label")

,group,label,explanation,text_block_en
12751,RUS,CRIME_FRAME_NEG,Die **[Gruppe 1]** wird explizit als Täterin d...,Zerstörter russischer Panzer\n\n„Zehn Zivilist...
14236,UKR,CRIME_FRAME_NEG,Die Gruppe wird explizit als potenzielle Quell...,In Belgorod und anderen Orten Russlands häufen...
6020,REFUG,CRIME_FRAME_NEG,Der Absatz verknüpft explizit die **illegale E...,Die illegalen Einreisen von [Gruppe 1] im Herb...
903,AZE,CRIME_FRAME_NEG,Der Absatz verknüpft die **[Gruppe 1]** expliz...,Im Mordprozess um den sogenannten Axtmörder vo...
13208,RUS,CRIME_FRAME_NEG,Der Absatz verknüpft explizit **[Gruppe 1]** m...,Die Firma soll demnach schon 2019 versucht hab...
...,...,...,...,...
6811,REFUG,NO_CRIME_FRAME,Der Absatz thematisiert ausschließlich gesundh...,Derzeit arbeite das Gesundheitsamt gemeinsam m...
6810,ROU,NO_CRIME_FRAME,Der Absatz thematisiert ausschließlich logisti...,\n\nSchwierigkeiten auf allen Wegen Ukrainisch...
6809,ESP,NO_CRIME_FRAME,"Der Absatz erwähnt weder Kriminalität, Sicherh...",Paris - Der Senegalese erhielt den Preis und w...
6816,REFUG,NO_CRIME_FRAME,Der Absatz erwähnt zwar Kriminalität (Festnahm...,Wegen Freundschaft mit Putin: Putin-Freund: Di...


In [49]:
# Show only crime frame cases for focused inspection
crime_cases = annotation_2_1[
    annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS"])
]

print(f"Crime frame cases: {len(crime_cases)}")

crime_cases[
    ["group", "label", "explanation", "text_block_en"]
].sort_values("label")

Crime frame cases: 856


,group,label,explanation,text_block_en
3,MIGR,CRIME_FRAME_NEG,Der Absatz verknüpft explizit die **[Gruppe 1]...,[Andere Gruppe] entsorgen eigene Papiere\n\nDi...
13273,ISLMST,CRIME_FRAME_NEG,Die Taliban werden explizit als **Terrorbewegu...,Die Taliban drohen den USA nach der Tötung des...
13341,AUT,CRIME_FRAME_NEG,Die Gruppe wird explizit als Täterin von Straf...,Eine 23-jährige [Gruppe 1] soll etliche PCR-Te...
13343,ASYL,CRIME_FRAME_NEG,Der Absatz verknüpft die **[Gruppe 1]** expliz...,Erinnerungen an den Fall in Freiburg\n\nDer „T...
13356,RUS,CRIME_FRAME_NEG,Die **[Gruppe 1]** wird explizit als Akteur da...,Wie schnell nach dem 24. Februar begannen die ...
...,...,...,...,...
11380,REFUG,CRIME_FRAME_POS,Der Absatz beschreibt präventive Maßnahmen (Ge...,Durch diese Kooperation seien seit der Gründun...
13675,REFUG,CRIME_FRAME_POS,Der Absatz thematisiert explizit **Präventions...,"Martin Dulig (SPD), Wirtschaftsminister von Sa..."
13705,ASYL,CRIME_FRAME_POS,Der Absatz verknüpft explizit die Gruppe mit *...,"Oft sind es politische Vorbilder, die einen in..."
377,MUSL,CRIME_FRAME_POS,Der Absatz verknüpft die Gruppe explizit mit *...,"Was könnte helfen, damit diese Männer sich bes..."


In [50]:
# Save one-shot candidates from Step 2.1

crime_neg_cases = annotation_2_1[annotation_2_1["label"] == "CRIME_FRAME_NEG"].copy()
crime_pos_cases = annotation_2_1[annotation_2_1["label"] == "CRIME_FRAME_POS"].copy()
unclear_cases = annotation_2_1[annotation_2_1["label"] == "UNCLEAR"].copy()

for data in [crime_neg_cases, crime_pos_cases, unclear_cases]:
    data["seed"] = RANDOM_SEED
    data["sample_size"] = SAMPLE_SIZE_2_1
    data["source_step"] = "2.1_one_shot"

if CRIME_FRAME_NEG.exists():
    old_neg = pd.read_csv(CRIME_FRAME_NEG)
    crime_neg_cases = pd.concat([old_neg, crime_neg_cases], ignore_index=True)

crime_neg_cases = crime_neg_cases.drop_duplicates(subset=["article_id", "par_index"])
crime_neg_cases.to_csv(CRIME_FRAME_NEG, index=False)

if CRIME_FRAME_POS.exists():
    old_pos = pd.read_csv(CRIME_FRAME_POS)
    crime_pos_cases = pd.concat([old_pos, crime_pos_cases], ignore_index=True)

crime_pos_cases = crime_pos_cases.drop_duplicates(subset=["article_id", "par_index"])
crime_pos_cases.to_csv(CRIME_FRAME_POS, index=False)

if UNCLEAR_FRAME.exists():
    old_unclear = pd.read_csv(UNCLEAR_FRAME)
    unclear_cases = pd.concat([old_unclear, unclear_cases], ignore_index=True)

unclear_cases = unclear_cases.drop_duplicates(subset=["article_id", "par_index"])
unclear_cases.to_csv(UNCLEAR_FRAME, index=False)

print(f"Saved CRIME_FRAME_NEG cases to {CRIME_FRAME_NEG}")
print(f"Saved CRIME_FRAME_POS cases to {CRIME_FRAME_POS}")
print(f"Saved UNCLEAR cases to {UNCLEAR_FRAME}")

Saved CRIME_FRAME_NEG cases to results/crime_frame_neg.csv
Saved CRIME_FRAME_POS cases to results/crime_frame_pos.csv
Saved UNCLEAR cases to results/unclear_frame.csv


## Step 2.2 — Check for Hard Cases (briseus equivalent)

This step runs the annotation **multiple times with higher temperature** (0.7) on a subset of rows. Higher temperature introduces randomness — if the model consistently agrees with itself across runs, the case is clear. If it disagrees, the case is hard and the instructions need clarification.

This is the Python equivalent of the `briseus()` function from the `bacchuss` R package.

After running, sort by agreement (lowest first) and inspect the hard cases:
- Why is the model uncertain?
- Can you add a decision rule to resolve the ambiguity?
- Are these genuinely borderline cases, or a problem with the instructions?

> **Note:** This runs `n_runs × sample_size` API calls. With n_runs=5 and sample_size=20, that is 100 calls. Adjust accordingly.

In [52]:
# ── Config ───────────────────────────────────────────────────────────────
SAMPLE_SIZE_2_2 = 2000
N_RUNS = 10
TEMPERATURE = 0.7

# Seed only for selecting NO_CRIME_FRAME rows!
NO_CRIME_FILL_SEED = 99

HARD_CASES_FRAME = RESULTS_DIR / "hard_cases.csv"
OUTPUT_2_2 = RESULTS_DIR / f"annotation_step2_2_fillseed{NO_CRIME_FILL_SEED}_n{SAMPLE_SIZE_2_2}_runs{N_RUNS}.csv"
# ─────────────────────────────────────────────────────────────────────────

# Include all CRIME_FRAME_NEG, CRIME_FRAME_POS, and UNCLEAR cases from Step 2.1
interesting_cases = annotation_2_1[
    annotation_2_1["label"].isin(["CRIME_FRAME_NEG", "CRIME_FRAME_POS", "UNCLEAR"])
].copy()

# Fill remaining slots with NO_CRIME_FRAME cases
n_fill = SAMPLE_SIZE_2_2 - len(interesting_cases)

if n_fill > 0:
    no_crime_candidates = annotation_2_1[
        annotation_2_1["label"] == "NO_CRIME_FRAME"
    ].copy()

    no_crime_fill = no_crime_candidates.sample(
        n=min(n_fill, len(no_crime_candidates)),
        random_state=NO_CRIME_FILL_SEED
    ).copy()

    sample_2_2 = pd.concat([interesting_cases, no_crime_fill], ignore_index=True)
else:
    no_crime_fill = pd.DataFrame()
    sample_2_2 = interesting_cases.copy()

sample_2_2 = sample_2_2.drop_duplicates(
    subset=["article_id", "par_index"]
).reset_index(drop=True)

print(f"Interesting cases included: {len(interesting_cases)}")
print(f"NO_CRIME_FRAME fill cases included: {len(no_crime_fill)}")
print(f"Sample size: {len(sample_2_2)} rows")
print(f"Runs per paragraph: {N_RUNS}")
print(f"Total API calls: {len(sample_2_2) * N_RUNS}")
print(f"Temperature: {TEMPERATURE}")
print(sample_2_2["label"].value_counts())

Interesting cases included: 856
NO_CRIME_FRAME fill cases included: 1144
Sample size: 1998 rows
Runs per paragraph: 10
Total API calls: 19980
Temperature: 0.7
label
NO_CRIME_FRAME     1144
CRIME_FRAME_NEG     837
CRIME_FRAME_POS      17
Name: count, dtype: int64


In [61]:
results_2_2 = []

for i, row in sample_2_2.iterrows():
    text = str(row["text_block_en"])
    run_labels = [] 
    run_explanations = []

    for run in range(N_RUNS):
        raw = annotate(text, instructions, reminder, temperature=TEMPERATURE)
        run_labels.append(parse_label(raw))
        run_explanations.append(parse_explanation(raw))

    label_counts = Counter(run_labels)
    majority_label, majority_count = label_counts.most_common(1)[0]
    agreement = majority_count / N_RUNS

    result = { 
        "article_id": row["article_id"],
        "par_index": row["par_index"],
        "group": row["group"],
        "text_block_en": text,
        "step2_1_label": row["label"],
        "final_label": majority_label,
        "agreement": agreement,
    }

    for r, lbl in enumerate(run_labels):
        result[f"run_{r+1}"] = lbl
        result[f"explanation_{r+1}"] = run_explanations[r]

    results_2_2.append(result)

    print(
        f"  [{i+1}/{len(sample_2_2)}] "
        f"{majority_label} "
        f"(agreement: {agreement:.0%}) | runs: {run_labels}"
    )

annotation_2_2 = pd.DataFrame(results_2_2).sort_values("agreement").reset_index(drop=True)

annotation_2_2.to_csv(OUTPUT_2_2, index=False)

print(f"\nStep 2.2 complete. Saved to {OUTPUT_2_2}")
print(annotation_2_2["final_label"].value_counts())

  [1/1998] CRIME_FRAME_NEG (agreement: 100%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [2/1998] CRIME_FRAME_NEG (agreement: 50%) | runs: ['CRIME_FRAME_NEG', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'NO_CRIME_FRAME']
  [3/1998] CRIME_FRAME_NEG (agreement: 100%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [4/1998] CRIME_FRAME_NEG (agreement: 100%) | runs: ['CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG', 'CRIME_FRAME_NEG']
  [5/1998] CRIME_FRAME_NEG (ag

### Inspect Hard Cases

Cases with **agreement < 80%** are hard cases — the model is uncertain and your instructions likely need a clearer decision rule for these.

For each hard case, ask:
1. What makes this paragraph ambiguous?
2. Can you write a decision rule that resolves the ambiguity?
3. Is this a genuine borderline case that needs a new label or sub-category?

Then go back to the **Annotation Instructions** cell above, update the instructions, and rerun both 2.1 and 2.2.

In [62]:
run_cols = [f"run_{r+1}" for r in range(N_RUNS)]
explanation_cols = [f"explanation_{r+1}" for r in range(N_RUNS)]

print("=== Full Results (sorted by agreement, lowest first) ===")

annotation_2_2[
    ["agreement", "step2_1_label", "final_label", "text_block_en"] 
    + run_cols 
    + explanation_cols
].sort_values("agreement")

=== Full Results (sorted by agreement, lowest first) ===


,agreement,step2_1_label,final_label,text_block_en,run_1,run_2,run_3,run_4,run_5,run_6,...,explanation_1,explanation_2,explanation_3,explanation_4,explanation_5,explanation_6,explanation_7,explanation_8,explanation_9,explanation_10
0,0.4,CRIME_FRAME_POS,CRIME_FRAME_POS,Bisher wurden in Deutschland 266.975 [Gruppe 1...,CRIME_FRAME_POS,NO_CRIME_FRAME,NO_CRIME_FRAME,CRIME_FRAME_NEG,CRIME_FRAME_POS,NO_CRIME_FRAME,...,**Label: CRIME_FRAME_POS**\n**Explanation:** D...,Der Absatz verknüpft die Gruppe **nicht expliz...,Der Absatz thematisiert zwar **Sicherheitsbede...,Der Absatz verknüpft die **[Gruppe 1]** expliz...,**Label:** **CRIME_FRAME_POS**\n**Explanation:...,Der Absatz spricht zwar **Sicherheitsbedenken*...,Der Absatz thematisiert explizit **Präventions...,Die Gruppe wird zwar im Kontext einer Diskussi...,Der Absatz thematisiert explizit **Präventions...,**Label:** **CRIME_FRAME_NEG**\n**Explanation:...
1,0.4,CRIME_FRAME_NEG,NO_CRIME_FRAME,Laut einer Antwort der Se­natsinnenverwaltung ...,NO_CRIME_FRAME,NO_CRIME_FRAME,CRIME_FRAME_POS,NO_CRIME_FRAME,CRIME_FRAME_POS,CRIME_FRAME_NEG,...,Die Gruppe wird zwar mit Kontrollen und Aufent...,Die Gruppe wird nicht explizit als Quelle von ...,**Label:** CRIME_FRAME_POS\n\n**Explanation:**...,Der Text verknüpft die Gruppe nicht direkt mit...,Der Absatz thematisiert explizit **Präventions...,**Label:** **CRIME_FRAME_NEG**\n**Explanation:...,**Label: CRIME_FRAME_NEG**\n**Explanation:** D...,Die Gruppe wird nicht explizit als Täter oder ...,Der Absatz beschreibt explizit präventive Maßn...,Der Text verbindet die Gruppe explizit mit Kri...
25,0.5,CRIME_FRAME_NEG,CRIME_FRAME_NEG,„Als die Polizei diese mit Spezialkräften durc...,NO_CRIME_FRAME,CRIME_FRAME_NEG,NO_CRIME_FRAME,NO_CRIME_FRAME,CRIME_FRAME_NEG,CRIME_FRAME_POS,...,Der Absatz erwähnt zwar eine Straftat (Messerf...,Der Absatz verknüpft die Gruppe explizit mit e...,Der Absatz führt zwar eine konkrete Straftat (...,Der Absatz verbindet zwar explizit ein Mitglie...,**Label:** **CRIME_FRAME_NEG**\n**Explanation:...,,,Die Gruppe [Gruppe 1] wird explizit als Teil d...,Die Gruppe wird zwar als Wohnbevölkerung einer...,*Der Absatz enthält sowohl eine explizite Verk...
26,0.5,CRIME_FRAME_NEG,NO_CRIME_FRAME,Berlin Der neue ukrainische Botschafter in Ber...,UNCLEAR,NO_CRIME_FRAME,NO_CRIME_FRAME,CRIME_FRAME_NEG,NO_CRIME_FRAME,CRIME_FRAME_NEG,...,**Der Absatz verknüpft [Gruppe 1] zwar mit ein...,**Label: NO_CRIME_FRAME**\n**Explanation:** De...,Der Absatz verknüpft **[Gruppe 1]** explizit n...,Der Absatz verknüpft **[Gruppe 1]** explizit m...,---\n**Label:** **NO_CRIME_FRAME**\n**Explanat...,**Label: CRIME_FRAME_NEG**\n**Explanation:** D...,Der Absatz verknüpft explizit **[Gruppe 1]** m...,Der Absatz verknüpft **[Gruppe 1]** explizit m...,Die im Absatz erwähnte **[Gruppe 1]** wird zwa...,Label: **UNCLEAR**\n\n**Explanation:**\nDer Pa...
27,0.5,CRIME_FRAME_NEG,NO_CRIME_FRAME,\n\nDie Mullahs haben erneut zwei [Gruppe 1] i...,NO_CRIME_FRAME,CRIME_FRAME_NEG,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,...,Die [Gruppe 1] wird nicht als Täterin oder mit...,Die **[Gruppe 1]** wird als Täterin des Vorwur...,**Label: NO_CRIME_FRAME**\n**Explanation:** Di...,Die Gruppe [Gruppe 1] wird nicht als Täterin o...,"Die Gruppe wird hier nicht als Täterin der ""ve...",Die Gruppe wird hier als festgenommene Verdäch...,Die **[Andere Gruppe]** wird direkt als Täteri...,Die [Gruppe 1] wird als Täterin der *versuchte...,Die **[Gruppe 1]** (Frau und Mann) wird expliz...,Die Absätze verknüpfen [Gruppe 1] (Mullahs) di...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
997,1.0,NO_CRIME_FRAME,NO_CRIME_FRAME,Der Post SV verkaufte sich keinesfalls schlech...,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,NO_CRIME_FRAME,...,Der Absatz behandelt ausschliesslich einen spo...,**Label: NO_CRIME_FRAME**\n**Explanation:** De...,**Label:** NO_CRIME_FRAME\n**Explanation:** De...,Der Absatz behandelt ausschl

In [63]:
hard_cases = annotation_2_2[annotation_2_2["agreement"] < 0.8].copy()

print(f"Hard cases (agreement < 80%): {len(hard_cases)} out of {len(annotation_2_2)}")

for _, row in hard_cases.iterrows():
    runs = [row[f"run_{r+1}"] for r in range(N_RUNS)]

    print(f"\n{'='*60}")
    print(f"Agreement: {row['agreement']:.0%}")
    print(f"Step 2.1 label: {row['step2_1_label']}")
    print(f"Final label: {row['final_label']}")
    print(f"Runs: {runs}")
    print(f"Text: {str(row['text_block_en'])[:500]}...")
    print("→ TODO: Why is this hard? What decision rule would resolve it?")

Hard cases (agreement < 80%): 266 out of 1998

Agreement: 40%
Step 2.1 label: CRIME_FRAME_POS
Final label: CRIME_FRAME_POS
Runs: ['CRIME_FRAME_POS', 'NO_CRIME_FRAME', 'NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS', 'NO_CRIME_FRAME', 'CRIME_FRAME_POS', 'NO_CRIME_FRAME', 'CRIME_FRAME_POS', 'CRIME_FRAME_NEG']
Text: Bisher wurden in Deutschland 266.975 [Gruppe 1] aus der Ukraine erfasst, gab das Bundesinnenministerium am Sonntag bekannt. [Andere Gruppe] müssen sich nicht bei den deutschen Behörden registrieren, weil sie für 90 Tage visumfrei einreisen können. Erfasst werden sie bei den Behörden erst, wenn sie sich dort melden, etwa um staatliche Hilfe in Anspruch zu nehmen.

Unter anderem der bayerische Innenminister Joachim Herrmann (CSU) hatte zuvor vor einer Sicherheitslücke bei der Aufnahme ukrainischer...
→ TODO: Why is this hard? What decision rule would resolve it?

Agreement: 40%
Step 2.1 label: CRIME_FRAME_NEG
Final label: NO_CRIME_FRAME
Runs: ['NO_CRIME_FRAME', 'NO_CRIME_

In [64]:
#check hardcases in detail! 

row = hard_cases.iloc[0] #checks first one needs to be changed if multiple

print("ARTICLE ID:", row["article_id"])
print("PAR INDEX:", row["par_index"])
print("GROUP:", row["group"])
print("AGREEMENT:", row["agreement"])
print("STEP 2.1 LABEL:", row["step2_1_label"])
print("FINAL LABEL:", row["final_label"])

print("\nTEXT:\n")
print(row["text_block_en"])

print("\nRUNS AND EXPLANATIONS:\n")
for r in range(N_RUNS):
    print(f"\n--- Run {r+1} ---")
    print("Label:", row[f"run_{r+1}"])
    print("Explanation:", row[f"explanation_{r+1}"])

ARTICLE ID: Tonline_added_1e21630211a305730c36fd6115990fa76b1cebe0_
PAR INDEX: 9
GROUP: REFUG
AGREEMENT: 0.4
STEP 2.1 LABEL: CRIME_FRAME_POS
FINAL LABEL: CRIME_FRAME_POS

TEXT:

Bisher wurden in Deutschland 266.975 [Gruppe 1] aus der Ukraine erfasst, gab das Bundesinnenministerium am Sonntag bekannt. [Andere Gruppe] müssen sich nicht bei den deutschen Behörden registrieren, weil sie für 90 Tage visumfrei einreisen können. Erfasst werden sie bei den Behörden erst, wenn sie sich dort melden, etwa um staatliche Hilfe in Anspruch zu nehmen.

Unter anderem der bayerische Innenminister Joachim Herrmann (CSU) hatte zuvor vor einer Sicherheitslücke bei der Aufnahme ukrainischer [Gruppe 1] gewarnt. "Es ist wichtig, dass durch erkennungsdienstliche Maßnahmen die Identifizierbarkeit der ankommenden Personen sichergestellt wird", sagte der Vorsitzende der Innenministerkonferenz den Zeitungen der Funke Mediengruppe.

Herrmann rief die anderen Bundesländer dazu auf, dem bayerischen Beispiel zu folge

In [65]:
# Agreement summary
print("=== Agreement Distribution ===")
print(annotation_2_2["agreement"].describe().round(2))

print(f"\nFull agreement (1.00): {len(annotation_2_2[annotation_2_2['agreement'] == 1.0])} rows")
print(f"Hard cases (<0.80):    {len(annotation_2_2[annotation_2_2['agreement'] < 0.8])} rows")
print(f"Very hard (<0.60):     {len(annotation_2_2[annotation_2_2['agreement'] < 0.6])} rows")

print("\n=== Final Label Distribution ===")
print(annotation_2_2["final_label"].value_counts())

=== Agreement Distribution ===
count    1998.00
mean        0.93
std         0.13
min         0.40
25%         0.90
50%         1.00
75%         1.00
max         1.00
Name: agreement, dtype: float64

Full agreement (1.00): 1493 rows
Hard cases (<0.80):    266 rows
Very hard (<0.60):     44 rows

=== Final Label Distribution ===
final_label
NO_CRIME_FRAME     1220
CRIME_FRAME_NEG     763
CRIME_FRAME_POS      15
Name: count, dtype: int64


In [66]:
# Save hard cases continuously, but replace old duplicates with the newest version
hard_cases["source_step"] = "2.2_multiple_runs"
hard_cases["n_runs"] = N_RUNS
hard_cases["temperature"] = TEMPERATURE
hard_cases["fill_seed"] = NO_CRIME_FILL_SEED

if HARD_CASES_FRAME.exists():
    old_hard_cases = pd.read_csv(HARD_CASES_FRAME)

    combined_hard_cases = pd.concat(
        [old_hard_cases, hard_cases],
        ignore_index=True
    )

    # keep the newest version when article_id + par_index already exists
    combined_hard_cases = combined_hard_cases.drop_duplicates(
        subset=["article_id", "par_index"],
        keep="last"
    )
else:
    combined_hard_cases = hard_cases.copy()

combined_hard_cases.to_csv(HARD_CASES_FRAME, index=False)

print(f"Saved hard cases to {HARD_CASES_FRAME}")
print(f"Total hard cases saved: {len(combined_hard_cases)}")

Saved hard cases to results/hard_cases.csv
Total hard cases saved: 323


## Next Steps

After inspecting the results:

1. **Note problems** — write down what went wrong and why in a markdown cell or comment
2. **Update instructions** — go back to the instructions cell and add/clarify decision rules
3. **Repeat 2.1 and 2.2** — rerun both steps with the updated instructions
4. **Stop when** your instructions broadly cover the cases you see, even if reliability is not yet perfect

Once coverage is broad, move to **Step 3** — testing on a curated sample of hard and soft cases with human gold-standard labels, adding chain-of-thought prompting and few-shot examples.

---

### Notes on This Round (fill in as you iterate)

| Round | Problem Identified | Decision Rule Added |
|-------|-------------------|---------------------|
| 1  (Seed 42, n= 200)   |Model is unsure whether CRIME_FRAME_NEG requires the group to be the actual perpetrator, or whether it is enough that the group praises, legitimizes, supports, or is linked to a terror attack. -> Should label as CRIME_FRAME_NEG| Zusätzliche Entscheidungsregel zu Terrorismus, Anschlägen und Gewaltlegitimation: Wenn eine Gruppe einen Anschlag, Terrorakt, Gewaltakt oder eine Straftat ausdrücklich begrüßt, legitimiert, feiert, rechtfertigt oder als Widerstand bzw. heroische Operation darstellt, vergeben Sie CRIME_FRAME_NEG, auch wenn die Gruppe keine direkte Verantwortung übernimmt oder nicht eindeutig als Täter genannt wird. Dies gilt auch dann, wenn die Gruppe eher als Unterstützer, Kommentator, ideologischer Bezugspunkt oder politischer Akteur erscheint, solange sie explizit mit Zustimmung, Rechtfertigung, Feier, Unterstützung oder ideologischer Nähe zu Terrorismus, politischer Gewalt oder Straftaten verbunden wird. Krieg, geopolitische Konflikte oder militärische Ereignisse sind nur dann CRIME_FRAME_NEG, wenn die Gruppe explizit mit Terrorismus, Anschlägen, Gewalt gegen Zivilisten, Straftaten oder Sicherheitsbedrohungen verbunden wird. |
| 2     |                   |                     |
| 3     |                   |                     |